# Which product is the model worst at?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [13]:
import os

import polars as pl
from google.cloud import bigquery, storage
from sklearn.metrics import average_precision_score, roc_auc_score

from fraud_detection.core.feature_contract.admission import load_admission_rules
from fraud_detection.core.promotion import parse_promotion_marker
from fraud_detection.core.schema import MODEL_INPUT_TABLE, SPLIT_TABLE
from fraud_detection.feature_engineering.derivations import apply_derivations
from fraud_detection.training.data import load_raw_split, prepare_features, to_lightgbm

PROJECT = os.environ["GCP_PROJECT_ID"]
bq = bigquery.Client(project=PROJECT)
gcs = storage.Client(project=PROJECT).bucket(f"{PROJECT}-models")

In [10]:
# The promoted model, read the way the scoring path reads it — the marker, never the
# newest artifact. Analysing a model nobody promoted would describe a decision nobody made.
import pickle

promoted = parse_promotion_marker(gcs.blob("promoted/production.json").download_as_text())
print(f"{promoted.run}  contract {promoted.contract_fingerprint}  code {promoted.code_version[:12]}")

bundle = pickle.loads(gcs.blob(f"lightgbm/{promoted.run}/model.pkl").download_as_bytes())
booster = bundle["booster"]

raw = load_raw_split(bq, PROJECT, "test", model_input_table=MODEL_INPUT_TABLE, split_table=SPLIT_TABLE)
derived = apply_derivations(raw, load_admission_rules().derivations)
features = prepare_features(derived)
scores = booster.predict(to_lightgbm(features.select(booster.feature_name())),
                         num_iteration=booster.best_iteration)

frame = raw.select(["TransactionID", "isFraud", "TransactionAmt", "TransactionDT",
                    "ProductCD", "card1", "addr1", "D1", "D9"]).with_columns(
    score=pl.Series(scores)
)
print(frame.shape)

ef077ec2  contract bdb97707da05adff  code 89145b690b7b
(59054, 10)


In [4]:
MIN_ROWS, MIN_POSITIVES = 500, 20


def by_segment(df: pl.DataFrame, column: str) -> pl.DataFrame:
    """PR-AUC within each level of `column`, with its own base rate beside it.

    Segments too small to estimate are reported as null rather than dropped: a segment
    nobody can measure is a finding about coverage, and silently omitting it would make
    the table look more complete than the data is.
    """
    rows = []
    for (level,), group in df.group_by([column], maintain_order=True):
        y, s = group["isFraud"].to_numpy(), group["score"].to_numpy()
        base = float(y.mean())
        measurable = len(group) >= MIN_ROWS and y.sum() >= MIN_POSITIVES
        pr = float(average_precision_score(y, s)) if measurable else None
        rows.append({
            column: level, "rows": len(group), "positives": int(y.sum()),
            "base_rate": round(base, 4),
            "pr_auc": None if pr is None else round(pr, 4),
            "lift_over_base": None if pr is None or base == 0 else round(pr / base, 2),
        })
    return pl.DataFrame(rows).sort("rows", descending=True)


overall = average_precision_score(frame["isFraud"].to_numpy(), frame["score"].to_numpy())
print(f"overall test PR-AUC {overall:.4f}, ROC-AUC "
      f"{roc_auc_score(frame['isFraud'].to_numpy(), frame['score'].to_numpy()):.4f}")

overall test PR-AUC 0.5308, ROC-AUC 0.8963


In [53]:
# Amount decile. The cost model prices a missed fraud at the transaction's full amount, so
# a model that is weak on the top decile is expensive in a way the headline metric hides.
banded = frame.with_columns(
    amt_decile=pl.col("TransactionAmt").qcut(10, labels=[f"{i}" for i in range(10)], allow_duplicates=True)
)

by_segment(banded, "amt_decile").sort("amt_decile")

amt_decile,rows,positives,base_rate,pr_auc,lift_over_base
str,i64,i64,f64,f64,f64
"""0""",5916,438,0.074,0.6422,8.67
"""1""",6301,218,0.0346,0.6469,18.7
"""2""",6701,222,0.0331,0.4869,14.7
"""3""",5368,94,0.0175,0.5074,28.98
"""4""",5248,164,0.0312,0.4549,14.56
"""5""",6803,254,0.0373,0.5413,14.5
"""6""",5001,100,0.02,0.4351,21.76
"""7""",5905,213,0.0361,0.482,13.36
"""8""",5967,185,0.031,0.4453,14.36


In [54]:
# Product, and hour of day. D9 is the hour as a fraction of one -- 24 distinct values --
# which is why it survives as a feature where the other D columns do not.
# print(by_segment(frame, "ProductCD"))
by_segment(frame, "ProductCD").sort("ProductCD")

ProductCD,rows,positives,base_rate,pr_auc,lift_over_base
str,i64,i64,f64,f64,f64
"""C""",6481,951,0.1467,0.7265,4.95
"""H""",1772,112,0.0632,0.4534,7.17
"""R""",2938,137,0.0466,0.8319,17.84
"""S""",2437,113,0.0464,0.7094,15.3
"""W""",45426,900,0.0198,0.213,10.75


In [ ]:
import plotly.express as px

# Auto-generated visualization for the table above
# Assuming the last output was a dataframe-like object
try:
    _plot_df = by_segment(frame, "ProductCD").sort("ProductCD")
    if hasattr(_plot_df, 'to_pandas'):
        _plot_df = _plot_df.to_pandas()
    
    fig = px.bar(_plot_df)
    fig.show()
except Exception as e:
    print(f"Failed to generate Plotly chart: {e}")
